In [1]:
import os
import datetime
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
 

In [2]:
creds = None
# The file token.json stores the user's access and refresh tokens, and is
# created automatically when the authorization flow completes for the first
# time.
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)
# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
        "credentials.json", SCOPES
        )
        creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
    with open("token.json", "w") as token:
        token.write(creds.to_json())

try:
    # Crear servicio Gmail
    service = build("gmail", "v1", credentials=creds)

    # 1. Obtener el último email (ordenado por fecha desc)
    response = service.users().messages().list(
        userId="me",
        maxResults=1,
        q="from: noreply@rico.com.vc has:attachment"  # puedes poner filtros como 'from:rico@...'
    ).execute()

    messages = response.get("messages", [])

    if not messages:
        print("No hay correos en la bandeja.")
    else:
        print("Último correo obtenido.")

    last_msg_id = messages[0]["id"]

    # 2. Obtener el contenido completo del mensaje
    msg = service.users().messages().get(
        userId="me",
        id=last_msg_id,
        format="full"
    ).execute()

    # 3. Buscar attachments dentro del payload
    def walk_parts(part):
        parts = []
        if part.get("parts"):
            for p in part["parts"]:
                parts.extend(walk_parts(p))
        else:
            parts.append(part)
        return parts

    all_parts = walk_parts(msg["payload"])

    # 4. Descargar los attachments
    for part in all_parts:
        filename = part.get("filename")
        body = part.get("body", {})

        if filename and body.get("attachmentId"):
            attachment_id = body["attachmentId"]

            # === ESTA ES LA LLAMADA EQUIVALENTE A TU HTTP GET ===
            att = service.users().messages().attachments().get(
                userId="me",
                messageId=last_msg_id,
                id=attachment_id
            ).execute()

            import base64, os
            file_data = base64.urlsafe_b64decode(att["data"])
            today = datetime.datetime.now().strftime("%Y-%m-%d")
            filename = f'trade_confirmation_{today}.pdf'

            # Guardar el archivo localmente
            os.makedirs("attachments", exist_ok=True)
            path = os.path.join("attachments", filename)

            with open(path, "wb") as f:
                f.write(file_data)

            print(f"Attachment descargado: {path}")

except Exception as e:
    print("Error:", e)



Último correo obtenido.
Attachment descargado: attachments/trade_confirmation_2025-11-22.pdf


In [3]:
import shutil
import pikepdf
from pathlib import Path
today = datetime.datetime.now().strftime("%Y-%m-%d")

attachments_dir = os.path.join(os.getcwd(), "attachments/trade_confirmation")
pdf_path = f'{attachments_dir}_{today}.pdf'

def decrypt_pdf(input_path, output_path, password):
    """Open password-protected PDF and save unencrypted copy."""
    with pikepdf.open(input_path, password=password) as pdf:
        pdf.save(output_path)

# Example
input_pdf = pdf_path
output_pdf = f'statement_unlocked_{today}.pdf'
password = "187"

decrypt_pdf(input_pdf, output_pdf, password)
print("Decrypted PDF saved:", output_pdf)

try: 
    os.remove(pdf_path)
    print(f"Removed encrypted PDF: {pdf_path}")
except Exception as e:
    print(f"Error removing file {pdf_path}: {e}")


src = os.path.join(os.getcwd(), output_pdf)
dst = os.path.join(os.getcwd(), "attachments")
shutil.move(src, dst)
print(f"Moved decrypted PDF to: {dst}")

Decrypted PDF saved: statement_unlocked_2025-11-22.pdf
Removed encrypted PDF: /Users/sergiobarrientoskellemberger/Documents/Scripts/Python/portfolio_dashboard/attachments/trade_confirmation_2025-11-22.pdf
Moved decrypted PDF to: /Users/sergiobarrientoskellemberger/Documents/Scripts/Python/portfolio_dashboard/attachments


In [4]:
import pdfplumber
import os

with pdfplumber.open(os.path.join(dst, output_pdf)) as pdf:
    first_page = pdf.pages[1]
    text = first_page.extract_text()
    print("Extracted text from first page:")
    print(text)



Extracted text from first page:
NOTA DE NEGOCIAÇÃO
Nr. nota Folha Data pregão
123718672 1 10/11/2025
RICO CORRETORA DE TITULOS E VALORES MOBILIARIOS S.A.
AV. PRESIDENTE JUSCELINO KUBITSCHEK, 1909 - TORRE NORTE 5 AND ITAIM 04538-132 SAO PAULO - SP
BIBI
Tel. +55 11 3003-5465, +55 11 4007-2465 Fax:
Internet: https://www.rico.com.br/ SAC: 0800-774-0402 e-mail:
C.N.P.J: 13.434.335/0001-60 Carta Patente:
Ouvidoria: Tel. 0800-722-3730 e-mail ouvidoria:
Cliente C.P.F./C.N.P.J/C.V.M./C.O.B.
15719775 SERGIO ALEJANDRO BARRIENTOS KELLEMBERGER 054.177.401-87
Rua 8A, 555 - Apto1 - Formosinha Tel. (0) 0-0000
Código cliente Assessor
73813-140 Formosa - GO 3-5 15719775 3534
Participante destino do repasse Cliente Valor Custodiante C.I
- 0 N
Banco Agência Conta corrente Acionista Administrador Complemento nome P. Vinc
N
Negociações
Negócios realizados
QNegociação C/V Tipo mercado Prazo Especificação do título Obs. (*) Quantidade Preço / Ajuste Valor Operação / Ajuste D/C
1-BOVESPA C FRACIONARIO BBSEGURI

In [5]:
import re

trades = re.findall(r"^1-BOVESPA.*$", text, flags=re.MULTILINE)
print(trades)


['1-BOVESPA C FRACIONARIO BBSEGURIDADE ON NM @ 17 34,17 580,89 D', '1-BOVESPA C VISTA BERKSHIRE DRN @ 8 132,19 1.057,52 D', '1-BOVESPA C FRACIONARIO BRASIL ON NM @ 25 22,78 569,50 D']


In [6]:
import pandas as pd 

# 2. Parse each line into structured data
records = []
for line in trades:
    parts = line.split()

    market = parts[0]                      # 1-BOVESPA
    side = parts[1]                        # C or V
    tipo = parts[2]                        # FRACIONARIO, VISTA, etc.

    # Extract description until '@'
    result = next((s for s in parts if s.isdigit()), None)
    index = parts.index(result)
    description = " ".join(parts[3:index -1])

    quantity = int(parts[index])

    # Convert Brazilian decimals 34,17 → 34.17
    price = float(parts[index + 1].replace(",", "."))
    total = float(parts[index + 2].replace(".", "").replace(",", "."))

    records.append({
        "market": market,
        "side": side,
        "type": tipo,
        "asset": description,
        "quantity": quantity,
        "price": price,
        "total": total,
    })

# 3. Convert to DataFrame
df = pd.DataFrame(records)
df


,market,side,type,asset,quantity,price,total
0,1-BOVESPA,C,FRACIONARIO,BBSEGURIDADE ON NM,17,34.17,580.89
1,1-BOVESPA,C,VISTA,BERKSHIRE DRN,8,132.19,1057.52
2,1-BOVESPA,C,FRACIONARIO,BRASIL ON NM,25,22.78,569.50


In [7]:
def extract_single_value(label, text):
    pattern = fr"{label}\s+([\d\.,]+)"
    match = re.search(pattern, text)
    return match.group(1) if match else None

total_cblc = extract_single_value("Total CBLC", text)
print("Total CBLC:", total_cblc)

total_bovespa = extract_single_value('Total Bovespa / Soma', text)
print('Total Bovespa', total_bovespa)

total_custos = extract_single_value('Total Custos / Despesas', text)
print('Total Custos', total_custos)


Total CBLC: 2.208,40
Total Bovespa 0,16
Total Custos 0,03


In [8]:
total_cblc = float(total_cblc.replace(".", "").replace(",", "."))
total_bovespa = float(total_bovespa.replace(".", "").replace(",", "."))
total_custos = float(total_custos.replace(".", "").replace(",", "."))

In [9]:
import re

def extract_first_date(text):
    pattern = r"\b(\d{2}/\d{2}/\d{4})\b"
    match = re.search(pattern, text)
    return match.group(1) if match else None

trade_date = extract_first_date(text)
trade_date

'10/11/2025'

In [10]:
final = df['total'].sum()
final

total_fees = total_cblc - final + total_custos + total_bovespa
total_fees

np.float64(0.6800000000002365)

In [27]:
df['fee'] = round(df['total'] / final * total_fees,2)
df['ticker'] = [pd.NA, pd.NA, pd.NA]
df

,date,side,ticker,asset,quantity,price,total,fee,portfolio
trade_id,,,,,,,,,
1f28462250e7483287d84ee4d01f5198,2025-11-10,BUY,<NA>,BBSEGURIDADE ON NM,17,34.17,580.89,0.18,G
765a327173474d8195785534b3a197bf,2025-11-10,BUY,<NA>,BERKSHIRE DRN,8,132.19,1057.52,0.33,G
5a2d28f82c43485f903c85578db818b3,2025-11-10,BUY,<NA>,BRASIL ON NM,25,22.78,569.50,0.18,G


In [28]:
import pandas as pd
import numpy as np
from typing import Union

# Define the placeholder used for unresolved tickers
UNRESOLVED_PLACEHOLDER = '!!!_INPUT_REQUIRED_!!!'

def resolve_tickers_in_df(df: pd.DataFrame, asset_column: str = 'asset', ticker_column: str = 'ticker') -> pd.DataFrame:
    """
    Scans the DataFrame to resolve missing ticker symbols by looking up other 
    entries for the same asset name within the same DataFrame. If no resolved
    ticker is found for an asset, it prompts the user for manual input.

    Args:
        df: The input DataFrame containing all trade data.
        asset_column: The name of the column containing the asset names (e.g., 'asset').
        ticker_column: The name of the column containing the ticker symbols (e.g., 'ticker').

    Returns:
        The updated DataFrame with resolved tickers where possible.
    """
    # Defensive copy to avoid modifying external state unexpectedly
    df_copy = df.copy() 
    
    # Identify unique asset names that currently have an unresolved placeholder or NaN
    unresolved_assets = df_copy[
        (df_copy[ticker_column] == UNRESOLVED_PLACEHOLDER) | df_copy[ticker_column].isna()
    ][asset_column].unique()

    assets_requiring_manual_input = []
    
    print(f"--- Starting Ticker Resolution for {len(unresolved_assets)} Assets ---")

    for asset_name in unresolved_assets:
        # 1. Look up resolved tickers for this asset name across the entire DF
        resolved_ticker_check = df_copy[
            (df_copy[asset_column] == asset_name) & 
            (df_copy[ticker_column] != UNRESOLVED_PLACEHOLDER) & 
            df_copy[ticker_column].notna()
        ][ticker_column].unique()

        if len(resolved_ticker_check) == 1:
            # 2. Case: Single, unique, resolved ticker found in the DF
            resolved_ticker = resolved_ticker_check[0]
            print(f"✅ Auto-resolved '{asset_name}' to '{resolved_ticker}'. Updating all entries.")
            
            # Update ALL rows for this asset, even those that were previously the placeholder
            df_copy.loc[df_copy[asset_column] == asset_name, ticker_column] = resolved_ticker
            
        elif len(resolved_ticker_check) > 1:
            # 3. Case: Conflict found (different resolved tickers for the same asset)
            # We choose the first one found for simplicity, but log the conflict.
            chosen_ticker = resolved_ticker_check[0]
            print(f"⚠️ Conflict detected for '{asset_name}'. Found {len(resolved_ticker_check)} tickers. Choosing '{chosen_ticker}'.")
            df_copy.loc[df_copy[asset_column] == asset_name, ticker_column] = chosen_ticker
            
        else:
            # 4. Case: Ticker is missing throughout the entire DF for this asset. Needs user input.
            assets_requiring_manual_input.append(asset_name)

    # 5. Handle assets requiring manual input using the input() function
    if assets_requiring_manual_input:
        print("\n--- MANUAL TICKER INPUT REQUIRED ---")
        print("Please provide the ticker symbol for the following unresolved assets.")
        
        # WARNING: The input() function might not work in all automated environments.
        # If this code fails due to missing input, you may need to run it in a
        # local Python terminal or notebook, or replace the input() call with 
        # a manual mapping update.
        
        for asset in assets_requiring_manual_input:
            # --- START INTERACTIVE INPUT ---
            try:
                # Prompt the user for input
                new_ticker = input(f"Enter ticker for '{asset}': ")
                
                # Check if the user entered anything
                if new_ticker.strip():
                    # Apply the new ticker to all rows for this asset
                    df_copy.loc[df_copy[asset_column] == asset, ticker_column] = new_ticker.strip().upper()
                    print(f"✅ Ticker '{new_ticker.strip().upper()}' assigned to '{asset}'.")
                else:
                    # If input is empty, revert to the placeholder
                    df_copy.loc[df_copy[asset_column] == asset, ticker_column] = UNRESOLVED_PLACEHOLDER
                    print(f"⚠️ No input provided for '{asset}'. Retaining placeholder.")
                    
            except EOFError:
                print(f"🛑 Input error (EOF) for '{asset}'. Retaining placeholder: {UNRESOLVED_PLACEHOLDER}")
                df_copy.loc[df_copy[asset_column] == asset, ticker_column] = UNRESOLVED_PLACEHOLDER
            # --- END INTERACTIVE INPUT ---
        
        print("----------------------------------\n")

    return df_copy



In [13]:
df = resolve_tickers_in_df(df)

--- Starting Ticker Resolution for 3 Assets ---

--- MANUAL TICKER INPUT REQUIRED ---
Please provide the ticker symbol for the following unresolved assets.
✅ Ticker 'G' assigned to 'BBSEGURIDADE ON NM'.
✅ Ticker 'G' assigned to 'BERKSHIRE DRN'.
✅ Ticker 'G' assigned to 'BRASIL ON NM'.
----------------------------------



In [14]:
df

,market,side,type,asset,quantity,price,total,fee,ticker
0,1-BOVESPA,C,FRACIONARIO,BBSEGURIDADE ON NM,17,34.17,580.89,0.18,G
1,1-BOVESPA,C,VISTA,BERKSHIRE DRN,8,132.19,1057.52,0.33,G
2,1-BOVESPA,C,FRACIONARIO,BRASIL ON NM,25,22.78,569.50,0.18,G


In [15]:
from datetime import datetime

trade_date
format_date = "%d/%m/%Y"

date_object = datetime.strptime(trade_date, format_date)
date_object

print(date_object.date())

df['date'] = date_object.date()
df

2025-11-10


,market,side,type,asset,quantity,price,total,fee,ticker,date
0,1-BOVESPA,C,FRACIONARIO,BBSEGURIDADE ON NM,17,34.17,580.89,0.18,G,2025-11-10
1,1-BOVESPA,C,VISTA,BERKSHIRE DRN,8,132.19,1057.52,0.33,G,2025-11-10
2,1-BOVESPA,C,FRACIONARIO,BRASIL ON NM,25,22.78,569.50,0.18,G,2025-11-10


In [16]:
current_cols = df.columns.tolist()
current_cols

new_order = ['date', 'side', 'ticker', 'asset', 'quantity', 'price','total','fee']

# 1. Define the mapping
mapping = {
    'C': 'BUY',
    'V': 'SELL'
}

# 2. Apply the mapping to the specific column
#    (The 'inplace=True' changes the DataFrame directly)
df['side'] = df['side'].replace(mapping)

df = df[new_order]
df['portfolio'] = [None, None, None]



/var/folders/sb/9vz_k9ms0tlfjj31q9q03hp00000gn/T/ipykernel_87930/249270893.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['portfolio'] = [None, None, None]


In [17]:
import pandas as pd
import numpy as np
from typing import Union

# Define the placeholder used for unresolved tickers
UNRESOLVED_PLACEHOLDER = '!!!_INPUT_REQUIRED_!!!'

def resolve_portfolio_in_df_per_trade(
    df: pd.DataFrame, 
    asset_column: str = 'asset', 
    portfolio_column: str = 'portfolio'
) -> pd.DataFrame:
    """
    Scans the DataFrame for missing portfolio assignments and prompts the user
    for input for *each row* where the assignment is blank. It does not look up
    previous assignments for the same asset, treating each blank entry as a 
    required manual input.

    Args:
        df: The input DataFrame containing all trade data.
        asset_column: The name of the column containing the asset names (e.g., 'asset').
        portfolio_column: The name of the column containing the assignments (e.g., 'portfolio').

    Returns:
        The updated DataFrame with resolved assignments where possible.
    """
    # Defensive copy to avoid modifying external state unexpectedly
    df_copy = df.copy() 
    
    # Identify the indices (rows) that require portfolio assignment
    unresolved_indices = df_copy[
        (df_copy[portfolio_column] == UNRESOLVED_PLACEHOLDER) | df_copy[portfolio_column].isna()
    ].index.tolist()

    if not unresolved_indices:
        print("✅ No portfolio assignments require manual input.")
        return df_copy
    
    print(f"--- MANUAL PORTFOLIO INPUT REQUIRED for {len(unresolved_indices)} Trades ---")
    print("Please provide the portfolio name for each transaction.")
    
    # Iterate through the indices that need resolution
    for idx in unresolved_indices:
        # Get the asset name for the current row
        asset_name = df_copy.loc[idx, asset_column]
        
        # --- START INTERACTIVE INPUT ---
        try:
            # Prompt the user for input
            # Displaying the index (row number) for context
            new_portfolio = input(f"[{idx}] Enter Portfolio for '{asset_name}': ")
            
            # Check if the user entered anything
            if new_portfolio.strip():
                # Apply the new portfolio to the single row
                df_copy.loc[idx, portfolio_column] = new_portfolio.strip().upper()
                print(f"✅ Portfolio '{new_portfolio.strip().upper()}' assigned to '{asset_name}' at row {idx}.")
            else:
                # If input is empty, revert to the placeholder
                df_copy.loc[idx, portfolio_column] = UNRESOLVED_PLACEHOLDER
                print(f"⚠️ No input provided for '{asset_name}' at row {idx}. Retaining placeholder.")
                
        except EOFError:
            print(f"🛑 Input error (EOF) for '{asset_name}' at row {idx}. Retaining placeholder: {UNRESOLVED_PLACEHOLDER}")
            df_copy.loc[idx, portfolio_column] = UNRESOLVED_PLACEHOLDER
        # --- END INTERACTIVE INPUT ---
        
    print("----------------------------------\n")

    return df_copy

In [18]:
df = resolve_portfolio_in_df_per_trade(df)

--- MANUAL PORTFOLIO INPUT REQUIRED for 3 Trades ---
Please provide the portfolio name for each transaction.
✅ Portfolio 'G' assigned to 'BBSEGURIDADE ON NM' at row 0.
✅ Portfolio 'G' assigned to 'BERKSHIRE DRN' at row 1.
✅ Portfolio 'G' assigned to 'BRASIL ON NM' at row 2.
----------------------------------



In [19]:
df

,date,side,ticker,asset,quantity,price,total,fee,portfolio
0,2025-11-10,BUY,G,BBSEGURIDADE ON NM,17,34.17,580.89,0.18,G
1,2025-11-10,BUY,G,BERKSHIRE DRN,8,132.19,1057.52,0.33,G
2,2025-11-10,BUY,G,BRASIL ON NM,25,22.78,569.50,0.18,G


In [6]:
import pandas as pd
import numpy as np
import uuid
from typing import Set

def add_validated_trade_id_for_missing_robust(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates unique UUIDs only for rows where 'trade_id' is missing (NaN/None),
    ensuring no collision and explicitly handling data types to prevent warnings.
    """
    df_copy = df.copy() 
    
    # 1. Ensure the 'trade_id' column exists, using 'object' dtype
    if 'trade_id' not in df_copy.columns:
        # Create the column and immediately set its type to 'object' (string)
        df_copy['trade_id'] = pd.Series([np.nan] * len(df_copy), dtype='object')
        print("💡 'trade_id' column created as type 'object'.")
    else:
        # If it exists, cast it to 'object' to handle any prior float/mixed inference
        df_copy['trade_id'] = df_copy['trade_id'].astype('object')
        
    # 2. Identify the rows where 'trade_id' is missing
    missing_mask = df_copy['trade_id'].isna()
    num_missing = missing_mask.sum()
    
    if num_missing == 0:
        print("✅ No missing 'trade_id' values found. DataFrame returned unchanged.")
        return df_copy
    
    # 3. Create a set of all currently existing, VALID IDs for collision check
    # Note: We can safely use .astype(str) here, as all IDs are non-NaN strings now.
    existing_ids: Set[str] = set(df_copy['trade_id'].dropna())
    
    print(f"Starting ID generation for {num_missing} missing rows. {len(existing_ids)} existing IDs found.")
    
    # 4. Generate new IDs 
    new_ids = []
    for _ in range(num_missing):
        unique_id = None
        while True:
            candidate_id = uuid.uuid4().hex
            if candidate_id not in existing_ids:
                unique_id = candidate_id
                existing_ids.add(unique_id)
                break
        new_ids.append(unique_id)
        
    # 5. Apply the new IDs only to the missing rows
    df_copy.loc[missing_mask, 'trade_id'] = new_ids

    print(f"✅ Successfully generated and assigned {num_missing} new, non-conflicting IDs.")
    return df_copy

In [23]:
df = add_validated_trade_id_for_missing_robust(df)
df

✅ No missing 'trade_id' values found. DataFrame returned unchanged.


,date,side,ticker,asset,quantity,price,total,fee,portfolio,trade_id
0,2025-11-10,BUY,G,BBSEGURIDADE ON NM,17,34.17,580.89,0.18,G,1f28462250e7483287d84ee4d01f5198
1,2025-11-10,BUY,G,BERKSHIRE DRN,8,132.19,1057.52,0.33,G,765a327173474d8195785534b3a197bf
2,2025-11-10,BUY,G,BRASIL ON NM,25,22.78,569.50,0.18,G,5a2d28f82c43485f903c85578db818b3


In [7]:
import pandas as pd 
from datetime import datetime

table_original = pd.read_excel('load_table_trades.xlsx',sheet_name='sheet1')
table_original 

table_original['total'] = table_original['total'].round(2)
table_original['price'] = table_original['price'].round(2)
table_original['fee'] = table_original['fee'].round(2)
table_original['quantity'] = table_original['quantity'].astype(int)


In [8]:
table_original
table_original['date'] = table_original['date'].dt.date
table_original

,trade_id,date,side,ticker,asset,quantity,price,total,fee,portfolio
0,NaN,2024-06-25,BUY,AAPL34,APPLE DRN,2,57.30,114.60,NaN,LEGEND
1,NaN,2024-07-24,BUY,OXYP34,OCCIDENT PTR DRN,3,56.45,169.35,0.05,LEGEND
2,NaN,2024-08-22,BUY,BERK34,BERKSHIRE DRN,3,124.39,373.17,0.11,LEGEND
3,NaN,2024-08-22,BUY,BOAC34,BANK AMERICA DRN,1,54.57,54.57,0.02,LEGEND
4,NaN,2024-09-30,BUY,AXPB34,AMERICAN EXP DRN,1,147.14,147.14,0.04,LEGEND
...,...,...,...,...,...,...,...,...,...,...
111,NaN,2025-03-25,BUY,VALE3,VALE ON NM,3,57.34,172.02,0.05,RENDA
112,NaN,2025-05-20,SELL,BBDC4,BRADESCO PN N1,7,15.50,108.50,0.02,RENDA
113,NaN,2025-11-05,SELL,CSMG3,COPASA ON NM,5,38.35,191.75,0.04,RENDA
114,NaN,2025-11-10,BUY,BBAS3,BRASIL ON EJ NM,25,22.78,569.50,0.18,RENDA


In [ ]:
final_table = add_validated_trade_id_for_missing_robust(table_original)
final_table

Starting ID generation for 116 missing rows. 0 existing IDs found.
✅ Successfully generated and assigned 116 new, non-conflicting IDs.


,trade_id,date,side,ticker,asset,quantity,price,total,fee,portfolio
0,db78c1263fa74a9da9688ad71a5c691d,2024-06-25,BUY,AAPL34,APPLE DRN,2,57.30,114.60,NaN,LEGEND
1,84071eba399a4a968380b81629222418,2024-07-24,BUY,OXYP34,OCCIDENT PTR DRN,3,56.45,169.35,0.05,LEGEND
2,bb06f3e80cac4524a99f9f0267a9848b,2024-08-22,BUY,BERK34,BERKSHIRE DRN,3,124.39,373.17,0.11,LEGEND
3,49c1acd5f8e540a3b5854b300344ea6b,2024-08-22,BUY,BOAC34,BANK AMERICA DRN,1,54.57,54.57,0.02,LEGEND
4,9101d965779e4a97a522e480d8f70291,2024-09-30,BUY,AXPB34,AMERICAN EXP DRN,1,147.14,147.14,0.04,LEGEND
...,...,...,...,...,...,...,...,...,...,...
111,31637d352a5f4461920ce66ddb5133a7,2025-03-25,BUY,VALE3,VALE ON NM,3,57.34,172.02,0.05,RENDA
112,61b04b63e3334f74b49443b8e9cb81e0,2025-05-20,SELL,BBDC4,BRADESCO PN N1,7,15.50,108.50,0.02,RENDA
113,cbf62090464a44eaabc2cc6457693677,2025-11-05,SELL,CSMG3,COPASA ON NM,5,38.35,191.75,0.04,RENDA
114,9225c1fc3c7c4dd994465d70f601e731,2025-11-10,BUY,BBAS3,BRASIL ON EJ NM,25,22.78,569.50,0.18,RENDA


In [27]:
final_table['fee'] = final_table['fee'].fillna(0)
final_table

,trade_id,date,side,ticker,asset,quantity,price,total,fee,portfolio
0,db78c1263fa74a9da9688ad71a5c691d,2024-06-25,BUY,AAPL34,APPLE DRN,2,57.30,114.60,0.00,LEGEND
1,84071eba399a4a968380b81629222418,2024-07-24,BUY,OXYP34,OCCIDENT PTR DRN,3,56.45,169.35,0.05,LEGEND
2,bb06f3e80cac4524a99f9f0267a9848b,2024-08-22,BUY,BERK34,BERKSHIRE DRN,3,124.39,373.17,0.11,LEGEND
3,49c1acd5f8e540a3b5854b300344ea6b,2024-08-22,BUY,BOAC34,BANK AMERICA DRN,1,54.57,54.57,0.02,LEGEND
4,9101d965779e4a97a522e480d8f70291,2024-09-30,BUY,AXPB34,AMERICAN EXP DRN,1,147.14,147.14,0.04,LEGEND
...,...,...,...,...,...,...,...,...,...,...
111,31637d352a5f4461920ce66ddb5133a7,2025-03-25,BUY,VALE3,VALE ON NM,3,57.34,172.02,0.05,RENDA
112,61b04b63e3334f74b49443b8e9cb81e0,2025-05-20,SELL,BBDC4,BRADESCO PN N1,7,15.50,108.50,0.02,RENDA
113,cbf62090464a44eaabc2cc6457693677,2025-11-05,SELL,CSMG3,COPASA ON NM,5,38.35,191.75,0.04,RENDA
114,9225c1fc3c7c4dd994465d70f601e731,2025-11-10,BUY,BBAS3,BRASIL ON EJ NM,25,22.78,569.50,0.18,RENDA


In [28]:
#Connection to SQL database

import psycopg2
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Connect to the database
try:
    connection = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )
    print("Connection successful!")
    
    # Create a cursor to execute SQL queries
    cursor = connection.cursor()
    
    # Example query
    cursor.execute("SELECT NOW();")
    result = cursor.fetchone()
    print("Current Time:", result)

    # Close the cursor and connection
    # cursor.close()
    # connection.close()
    # print("Connection closed.")

except Exception as e:
    print(f"Failed to connect: {e}")



Connection successful!
Current Time: (datetime.datetime(2025, 11, 22, 20, 42, 22, 276699, tzinfo=datetime.timezone.utc),)


In [22]:
import psycopg2
import os

# --- Configuration (Use your environment variables) ---
DB_HOST = os.environ.get("host")
DB_NAME = os.environ.get("dbname")
DB_USER = os.environ.get("user")
DB_PASSWORD = os.environ.get("password")
DB_PORT = 5432

connection = None
table_names = []

try:
    # 1. Establish the connection
    connection = psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT
    )
    
    with connection.cursor() as cur:
        # 2. Define the SQL query to fetch table names
        # This query selects table_name from the system metadata view.
        # It filters:
        # - table_schema = 'public' (to see only your tables, not system tables)
        # - table_type = 'BASE TABLE' (to exclude views and temporary tables)
        sql_query = """
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public' 
        AND table_type = 'BASE TABLE';
        """
        
        # 3. Execute the query
        cur.execute(sql_query)
        
        # 4. Fetch the results
        # Each row will be a tuple containing the table name (e.g., ('trades',))
        table_rows = cur.fetchall()
        
        # 5. Extract the table names into a clean list
        table_names = [row[0] for row in table_rows]
        
        print("--- 📚 Available Tables for Analysis ---")
        if table_names:
            for name in table_names:
                print(f"➡️ {name}")
        else:
            print("No user-created tables found in the 'public' schema.")
            
except Exception as e:
    print(f"❌ Database connection or query error: {e}")

finally:
    # Always close the connection
    if connection:
        connection.close()

# --- Next Step ---
if table_names:
    print("\n--- Next Step: Pulling Data ---")
    # You can now iterate through this list to pull data from each table
    print(f"The first discovered table is: **{table_names[0]}**")

--- 📚 Available Tables for Analysis ---
➡️ trades

--- Next Step: Pulling Data ---
The first discovered table is: **trades**


In [29]:
from io import StringIO

buffer = StringIO()

final_table.to_csv(buffer, header=False, index=False)

buffer.seek(0)

columns = final_table.columns.to_list()

try:
    with connection.cursor() as cur:
        cur.copy_from(
            file=buffer,
            table = 'trades',
            sep=",",
            columns=columns
        )

        connection.commit()
        print(f'Successfully inserted {len(final_table)} rows into trades table')

except Exception as e:
    connection.rollback()
    print(f'Database error: {e}')

finally:
    cursor.close()
    connection.close()

Successfully inserted 116 rows into trades table
